# Lesson 22 Lab — A Reproducible Single-Node Container

**Puzzle:** What must a container specification pin beyond the vLLM image tag?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

A container packages user space, but it still depends on host driver/runtime integration, GPU visibility, shared memory, model/cache mounts, secrets, health probes, and a rollback image digest.


## 0. Predict before running

1. Find every mutable identifier in the draft spec.
2. Check that model files are mounted read-only.
3. Name the host-level test still required.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The configuration audit builds a Docker deployment manifest, validates digest pinning, read-only model mounts, cache separation, IPC/shared-memory choice, secret handling, health checks, and resource limits. Docker execution is explicitly absent on the remote training container.

- The image does not contain the host GPU driver.
- A floating tag is not an immutable release identity.
- Writable cache, model, logs, and secrets have different lifecycle rules.


## 2. Derive the mechanism

The NVIDIA container runtime passes devices and driver libraries into an image. vLLM may use shared memory for tensor-parallel communication; model caches should persist outside the writable layer. Immutable image digests and model hashes allow rollback, while API keys should enter through a secret mechanism rather than command arguments.

### Mechanism at a glance

```mermaid
flowchart LR
  I["image digest"] --> C["container spec"]
  M["model hash + read-only mount"] --> C
  S["secret injection"] --> C
  G["GPU runtime + shared memory"] --> C
  C --> H["health + generation test"]
  H --> R["promote or rollback digest"]
```

### Walk it step by step

1. **Pin immutable inputs.** Use image digest, model hash, and explicit arguments.
2. **Separate mounts.** Make model read-only and cache/log destinations intentional.
3. **Inject runtime concerns.** Configure GPU access, shared memory, ports, and secrets.
4. **Test on a clean host.** Exercise health, generation, restart, and rollback.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 22
LESSON_TITLE = 'A Reproducible Single-Node Container'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260834
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | an unpinned `latest` image and implicit volumes |
| Candidate | digest-pinned image, explicit GPU/runtime, mounts, health, secrets, and rollback |
| Held constant | one model manifest, serving arguments, port, and security policy |
| Measurements | passed checks, failed checks, image pinning, mount modes, health command, and native Docker status |
| Evidence | `compatibility-probe` |

**Experiment:** Generate and lint a single-node container manifest against twelve deployment invariants.


## 5. Inspect the experiment code

The notebook represents the deployment as data and evaluates named checks. It does not call Docker when the daemon is outside the environment, so no container-runtime success is implied.

Do not execute until the code matches the frozen table.


In [2]:
manifest={"image":"vllm/vllm-openai@sha256:"+"a"*64,"gpu":"all","ipc":"host","port":8000,
 "mounts":[{"source":"/srv/models/qwen","target":"/models/qwen","mode":"ro","kind":"model"},
           {"source":"/srv/cache/vllm","target":"/cache/vllm","mode":"rw","kind":"cache"}],
 "secret":{"source":"docker-secret","name":"vllm_api_key","in_command":False},
 "health":{"path":"/health","interval_s":10,"start_period_s":180},
 "command":["--model","/models/qwen","--max-model-len","8192","--disable-log-requests"],
 "rollback_image":"vllm/vllm-openai@sha256:"+"b"*64}
checks={"image_digest":"@sha256:" in manifest["image"],"rollback_digest":"@sha256:" in manifest["rollback_image"],
 "gpu_explicit":bool(manifest["gpu"]),"ipc_explicit":manifest["ipc"] in {"host","private"},
 "model_read_only":any(x["kind"]=="model" and x["mode"]=="ro" for x in manifest["mounts"]),
 "cache_separate":any(x["kind"]=="cache" and x["mode"]=="rw" for x in manifest["mounts"]),
 "secret_external":manifest["secret"]["source"] in {"docker-secret","file","env-file"},
 "secret_not_command":not manifest["secret"]["in_command"],"health_path":manifest["health"]["path"]=="/health",
 "startup_budget":manifest["health"]["start_period_s"]>=120,"port_explicit":isinstance(manifest["port"],int),
 "request_logging_disabled":"--disable-log-requests" in manifest["command"]}
metrics={"manifest":manifest,"checks":checks,"checks_passed":sum(checks.values()),"checks_total":len(checks),
 "image_digest_pinned":checks["image_digest"],"model_read_only":checks["model_read_only"],
 "secret_external":checks["secret_external"],"native_docker_executed":False}
analysis=(f"The manifest passed {metrics['checks_passed']}/{metrics['checks_total']} static invariants, "
          "including digest pinning, read-only model bytes, external secrets, and startup-aware health. "
          "No Docker daemon was invoked.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Checks passed | 12 |
| Checks total | 12 |
| Image digest pinned | yes |
| Model read-only | yes |
| Secret external | yes |
| Native Docker executed | no |


## 7. Explain the result

The manifest passed 12/12 static invariants, including digest pinning, read-only model bytes, external secrets, and startup-aware health. No Docker daemon was invoked.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The installed package/API/configuration surface was inspected. Availability or lint success is not equivalent to native feature execution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 22, "title": 'A Reproducible Single-Node Container', "environment": ENV,
    "evidence_label": 'compatibility-probe', "metrics": metrics,
    "analysis": analysis, "conclusion": 'The audited manifest closes common reproducibility and secret-handling gaps; actual container execution remains a separate host test.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 22,
  "title": "A Reproducible Single-Node Container",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260834
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "manifest": {
      "image": "vllm/vllm-openai@sha256:aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa",
      "gpu": "all",
      "ipc": "host",
      "port": 8000,
      "mounts": [
        {
          "source": "/srv/models/qwen",
          "target": "/models/qwen",
          "mode": "ro",
          "kind": "model"
        },
        {
          "source": "/srv/cache/vllm",
          "target": "/cache/vllm",
          "mode": "rw",
          "kind": "cache"
        }
      ],
      "secret": {
        "source": "docker-secret",
        "name": "vllm_api_key",
        "in_command":

## 9. Make the bounded decision

> The audited manifest closes common reproducibility and secret-handling gaps; actual container execution remains a separate host test.

**Acceptance/rollback:** Promote the container spec only after lint gates and a cold host start complete generation, health, restart, and rollback tests.

**Failure analysis:** Static lint cannot verify host driver compatibility, pull permissions, runtime hooks, actual shared-memory needs, or cold-start duration.


## 10. Extend the evidence

Run the digest on a clean GPU host, verify model hashes, send traffic, restart during load, rotate the secret, and roll back to the previous digest.

The full boundary and references are in [`README.md`](README.md).
